# Aula 15 — Retrieval, Semantic Search and Grounding

A Aula 14 mostrou como modelos generativos produzem texto. Agora vamos separar outra capacidade:

> **Como localizar informação relevante antes de gerar uma resposta?**

Nesta aula, retrieval e generation permanecem separados. **RAG ainda não será construído.**

## Objetivos

Ao final, você deverá conseguir:

- explicar retrieval;
- distinguir busca lexical de busca semântica;
- calcular similaridade cosseno;
- interpretar ranking e top-k;
- explicar chunking;
- construir um evidence pack;
- explicar grounding;
- identificar failure modes de retrieval.

## Glossário da aula

Esta aula reutiliza conceitos já existentes — TF-IDF, embedding, cosine similarity, document, corpus e evidence — e introduz termos que serão incorporados ao Glossário Vivo antes da promoção para student-ready.


## 1. Da geração para retrieval

Na Aula 14:

```text
contexto
→ modelo generativo
→ próximo token
→ resposta
```

Agora isolamos outra etapa:

```text
query
→ coleção
→ comparação
→ score
→ ranking
→ evidência recuperada
```

A diferença é fundamental:

```text
retrieval
→ encontra informação

generation
→ produz informação em forma de saída
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Ambiente da Aula 15 pronto.")
print("Internet OFF: os experimentos usam dados locais e determinísticos.")


## 2. Query, corpus, score e ranking

Vamos usar um corpus pequeno e transparente.

Cada documento contém uma informação diferente. A consulta será comparada com todos eles.

O objetivo não é produzir um benchmark; é tornar o mecanismo de retrieval observável.


In [ ]:
documents = pd.DataFrame([
    {"doc_id": "D1", "text": "O sistema permite redefinir a senha pela página de login."},
    {"doc_id": "D2", "text": "O prazo para reembolso é de até cinco dias úteis."},
    {"doc_id": "D3", "text": "A autenticação em dois fatores aumenta a segurança da conta."},
    {"doc_id": "D4", "text": "A equipe financeira processa devoluções após a aprovação da solicitação."},
    {"doc_id": "D5", "text": "O usuário pode alterar o endereço de e-mail nas configurações do perfil."},
])

display(documents)


## 3. Lexical retrieval com TF-IDF

Na busca lexical, consulta e documentos são comparados principalmente pela presença e importância dos termos.

Vamos representar corpus + query com TF-IDF e calcular cosine similarity.


In [ ]:
def lexical_search(query, documents, top_k=3):
    vectorizer = TfidfVectorizer(lowercase=True)
    matrix = vectorizer.fit_transform(documents["text"].tolist() + [query])

    doc_matrix = matrix[:-1]
    query_vector = matrix[-1]

    scores = cosine_similarity(query_vector, doc_matrix).ravel()

    result = documents.copy()
    result["score"] = scores
    result = result.sort_values("score", ascending=False).reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))

    return result.head(top_k)

query_lexical = "como solicitar reembolso"
lexical_result = lexical_search(query_lexical, documents, top_k=5)
display(lexical_result)


### Observe

O ranking transforma uma coleção em uma ordem relativa para uma consulta específica.

Um score alto significa apenas que, segundo aquele método de representação e comparação, o item ficou mais próximo da consulta.

Isso **não garante** que:

- o documento responda completamente;
- esteja atualizado;
- seja factual;
- seja suficiente para uma decisão.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lexical_result["doc_id"], lexical_result["score"])
ax.set(
    xlabel="Documento",
    ylabel="Cosine similarity",
    title=f"Ranking lexical — query: {query_lexical!r}"
)
ax.grid(axis="y", alpha=.25)
plt.show()
plt.close(fig)


## 4. Semantic search

Busca semântica tenta comparar significados por meio de representações vetoriais densas.

Para manter a aula **Internet OFF** e totalmente reproduzível, vamos usar vetores didáticos pré-computados.

> **Importante:** estes vetores são proxies pedagógicos. Não são embeddings produzidos por um modelo real e não constituem benchmark.

A intenção é observar a geometria do retrieval.


In [ ]:
semantic_items = pd.DataFrame([
    {"item": "recuperar senha", "x": 0.95, "y": 0.10},
    {"item": "acesso à conta", "x": 0.88, "y": 0.18},
    {"item": "esqueci minha credencial", "x": 0.91, "y": 0.14},
    {"item": "devolução de pagamento", "x": 0.08, "y": 0.93},
    {"item": "alterar e-mail", "x": 0.35, "y": 0.55},
])

query_embedding = np.array([[0.93, 0.12]], dtype=float)
item_embeddings = semantic_items[["x", "y"]].to_numpy(dtype=float)

semantic_scores = cosine_similarity(query_embedding, item_embeddings).ravel()

semantic_result = semantic_items[["item"]].copy()
semantic_result["score"] = semantic_scores
semantic_result = semantic_result.sort_values("score", ascending=False).reset_index(drop=True)
semantic_result.insert(0, "rank", np.arange(1, len(semantic_result) + 1))

display(semantic_result)


### Por que isso é diferente da busca lexical?

Expressões como:

- “recuperar senha”;
- “acesso à conta”;
- “esqueci minha credencial”;

podem ser tratadas como semanticamente próximas mesmo quando não usam exatamente as mesmas palavras.

Em sistemas reais, os vetores seriam produzidos por um embedding model. Aqui, apenas simulamos posições vetoriais para estudar o mecanismo.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(semantic_items["x"], semantic_items["y"])

for _, row in semantic_items.iterrows():
    ax.annotate(row["item"], (row["x"], row["y"]), xytext=(5, 5), textcoords="offset points")

ax.scatter(query_embedding[0, 0], query_embedding[0, 1], marker="x", s=120)
ax.annotate("query", (query_embedding[0, 0], query_embedding[0, 1]), xytext=(5, -15), textcoords="offset points")

ax.set(xlabel="Dimensão didática 1", ylabel="Dimensão didática 2", title="Espaço semântico didático")
ax.grid(alpha=.25)
plt.show()
plt.close(fig)


## 5. Top-k de retrieval

Em retrieval, **top-k** significa quantos resultados melhor ranqueados serão devolvidos.

Isso é diferente do top-k da Aula 14:

```text
top-k em geração
→ limita tokens candidatos

top-k em retrieval
→ limita itens recuperados
```

O mesmo nome aparece em mecanismos diferentes.


In [ ]:
for k in [1, 2, 3]:
    print(f"Top-{k}")
    display(lexical_search(query_lexical, documents, top_k=k)[["rank", "doc_id", "text", "score"]])


## 6. Chunking

Documentos longos podem conter múltiplos assuntos. Indexar o documento inteiro pode diluir a relevância de um trecho específico.

**Chunking** divide o conteúdo em unidades menores para recuperação.

Trade-off:

```text
chunks muito grandes
→ mais contexto, menos precisão local

chunks muito pequenos
→ mais precisão local, risco de perder contexto
```


In [ ]:
long_document = (
    "A conta pode usar autenticação em dois fatores. "
    "A senha pode ser redefinida pela página de login. "
    "O prazo para reembolso é de até cinco dias úteis. "
    "O endereço de e-mail pode ser alterado no perfil."
)

chunks = [s.strip() + "." for s in long_document.split(".") if s.strip()]
chunk_df = pd.DataFrame({
    "chunk_id": [f"C{i+1}" for i in range(len(chunks))],
    "text": chunks
})

query_chunk = "qual é o prazo do reembolso"
chunk_result = lexical_search(query_chunk, chunk_df.rename(columns={"chunk_id": "doc_id"}), top_k=len(chunk_df))
chunk_result = chunk_result.rename(columns={"doc_id": "chunk_id"})

display(chunk_result)


## 7. Grounding e evidence pack

Agora vamos montar um pequeno **evidence pack**.

Grounding significa vincular a resposta ou decisão à evidência disponível.

Nesta aula, não usaremos LLM. A resposta será construída deterministicamente a partir do trecho mais bem ranqueado.

Isso é intencional:

```text
retrieval
→ evidência

grounding
→ restrição à evidência

generation
→ ainda não
```


In [ ]:
best_chunk = chunk_result.iloc[0]

evidence_pack = {
    "query": query_chunk,
    "evidence_id": best_chunk["chunk_id"],
    "evidence_text": best_chunk["text"],
    "retrieval_score": float(best_chunk["score"]),
}

grounded_answer = (
    f"Com base em {evidence_pack['evidence_id']}: "
    f"{evidence_pack['evidence_text']}"
)

display(pd.DataFrame([evidence_pack]))
print(grounded_answer)


## 8. Failure modes de retrieval

Retrieval pode falhar antes que qualquer modelo gere uma resposta.

Exemplos:

- query ambígua;
- termos incompatíveis entre consulta e documentos;
- embedding inadequado;
- chunk mal escolhido;
- documento correto ausente;
- ranking ruim;
- top-k pequeno demais;
- top-k grande demais;
- evidências contraditórias;
- documento desatualizado.

Portanto:

> **grounding só pode ser tão bom quanto a evidência disponível e recuperada.**


## 9. Exercício 1 — Ranking lexical

Use a consulta:

`"segurança da conta"`

Execute `lexical_search()` e descubra os dois documentos mais bem ranqueados.


In [ ]:
# Sua resposta aqui


### Dica

Use:

```python
lexical_search(query, documents, top_k=2)
```


In [ ]:
# Solução executável
exercise_query = "segurança da conta"
exercise_result = lexical_search(exercise_query, documents, top_k=2)
display(exercise_result)


## 10. Exercício 2 — Lexical vs semantic

Considere as expressões:

- “recuperar senha”;
- “esqueci minha credencial”.

Explique por que uma busca lexical pode enxergar pouca sobreposição entre elas enquanto uma representação semântica pode aproximá-las.

Sua resposta deve mencionar **representação**, não apenas “inteligência do modelo”.


### Resposta de referência

Busca lexical depende fortemente dos termos presentes na consulta e no documento. As duas expressões usam palavras diferentes.

Busca semântica compara representações vetoriais que podem colocar expressões com significado relacionado em regiões próximas do espaço vetorial, mesmo sem sobreposição literal.

Isso não torna semantic search automaticamente superior: o resultado depende da qualidade da representação e da tarefa.


## 11. Exercício 3 — Chunking

Use `chunk_df` e procure por:

`"como alterar meu e-mail"`

Identifique o chunk mais bem ranqueado.


In [ ]:
# Sua resposta aqui


### Dica

Reutilize `lexical_search()`. Para isso, renomeie temporariamente `chunk_id` para `doc_id`.


In [ ]:
# Solução executável
email_chunks = chunk_df.rename(columns={"chunk_id": "doc_id"})
email_result = lexical_search("como alterar meu e-mail", email_chunks, top_k=1)
display(email_result)


## 12. Exercício 4 — Grounding

Considere este evidence pack:

```text
C1: O prazo para reembolso é de até cinco dias úteis.
C2: A senha pode ser redefinida pela página de login.
```

Pergunta:

**Qual é o prazo para reembolso?**

Responda usando apenas a evidência e indique o trecho que sustenta sua resposta.

Depois reflita: o que você deveria fazer se nenhum dos trechos contivesse a informação necessária?


### Resposta de referência

**Resposta:** o prazo para reembolso é de até cinco dias úteis.

**Evidência:** C1.

Se nenhum trecho sustentasse a resposta, o sistema deveria evitar inventar informação e sinalizar ausência de evidência suficiente, ampliar a busca ou escalar o caso.


## 13. Reprodutibilidade

Esta versão:

- usa Internet OFF;
- não usa API externa;
- não usa vector database;
- não usa LLM;
- executa TF-IDF e cosine similarity reais;
- usa vetores semânticos sintéticos explicitamente didáticos;
- não apresenta os vetores como benchmark.

Uma futura comparação entre embedding models reais deverá ser produzida na trilha **AUTHOR / EVIDENCE**.


## 14. Síntese

Você deve sair desta aula distinguindo:

```text
retrieval
→ encontra evidência

ranking
→ ordena candidatos

chunking
→ define unidades recuperáveis

grounding
→ vincula resposta à evidência

generation
→ produz a saída
```

### Próxima ponte

Agora temos duas capacidades separadas:

```text
Aula 14
→ geração

Aula 15
→ retrieval + grounding
```

A próxima pergunta é:

> **Como combinar retrieval e generation em um único sistema sem perder rastreabilidade da evidência?**

Essa será a entrada para a **Aula 16 — Retrieval-Augmented Generation (RAG)**.
